# Fine-tune Qwen2.5-Coder for Indonesian → ClickHouse Text-to-SQL

**Runtime → Change runtime type → T4 GPU**, then run top to bottom.

Upload `colab_upload.zip` when asked (contains `gemini-code.md`, `seed.sql`, `train.jsonl`, `eval.jsonl`).

Output: `text2sql-q4_k_m.gguf` + `Modelfile` for Ollama (~2 GB for 3B, ~1 GB for 1.5B).

In [1]:
!pip install -q unsloth chdb

In [2]:
import os, zipfile
from google.colab import files
uploaded = files.upload()  # pick colab_upload.zip
zipfile.ZipFile(next(iter(uploaded))).extractall()
assert all(os.path.exists(f) for f in ["gemini-code.md", "seed.sql", "train.jsonl", "eval.jsonl"]), os.listdir()

Saving colab_upload.zip to colab_upload.zip


In [3]:
MODEL = "unsloth/Qwen2.5-Coder-3B-Instruct"  # ~2 GB as Q4; or unsloth/Qwen2.5-Coder-1.5B-Instruct (~1 GB)
MAX_LEN = 2048
EPOCHS = 3  # eval loss per epoch is printed during training; it was still falling at epoch 2 last time

## Load model + LoRA

In [4]:
from unsloth import FastLanguageModel, is_bfloat16_supported

model, tokenizer = FastLanguageModel.from_pretrained(MODEL, max_seq_length=MAX_LEN, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.8: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth 2026.10.8 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).


## Data

In [5]:
import json
from datasets import Dataset

def load(path):
    return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]

train_rows, eval_rows = load("train.jsonl"), load("eval.jsonl")
to_ds = lambda rows: Dataset.from_list([{"text": tokenizer.apply_chat_template(r["messages"], tokenize=False)} for r in rows])
train_ds, eval_ds = to_ds(train_rows), to_ds(eval_rows)
longest = max(len(tokenizer(t).input_ids) for t in train_ds["text"])
print(len(train_rows), "train,", len(eval_rows), "eval, longest sample", longest, "tokens")
assert longest < MAX_LEN

4256 train, 134 eval, longest sample 669 tokens


## Evaluation: execution accuracy on embedded ClickHouse

Builds the same schema + seed data with `chdb`, then a generated query counts as correct when it returns the same rows as the reference query.

In [6]:
import re
import json
from chdb import session as chs

spec = open("gemini-code.md", encoding="utf-8").read()
ddl = re.search(r"```sql\n(.*?)```", spec, re.S).group(1)

def statements(text):
    text = re.sub(r"--[^\n]*", "", text)
    return [s.strip() for s in text.split(";") if s.strip()]

ch = chs.Session()
ch.query("DROP DATABASE IF EXISTS crm")  # safe to re-run this cell
ch.query("CREATE DATABASE crm")
ch.query("USE crm")
for s in statements(ddl) + statements(open("seed.sql", encoding="utf-8").read()):
    ch.query(s)
ch.query("SET readonly = 1")  # generated SQL can't modify the data
print(ch.query("SELECT count() FROM deals", "CSV"))

def run(sql):
    """Rows as a sorted list (order-insensitive), floats rounded; None if the query fails."""
    try:
        b = ch.query(sql, "JSONCompact").bytes()
        if not b:  # some chdb versions return an empty result instead of raising on bad SQL
            return None
        data = json.loads(b)["data"]
    except Exception:  # invalid SQL from the model or JSON decoding failure
        return None
    canon = lambda row: json.dumps([round(v, 4) if isinstance(v, float) else v for v in row])
    return sorted(map(canon, data))  # ponytail: ignores ORDER BY; compare in order if ranking accuracy matters

def clean(text):
    m = re.search(r"```(?:sql)?\s*(.*?)```", text, re.S)
    return (m.group(1) if m else text).strip().rstrip(";")

def generate_sql(messages):
    prompt = tokenizer.apply_chat_template(messages[:2], tokenize=False, add_generation_prompt=True)
    ids = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**ids, max_new_tokens=512, do_sample=False)
    return clean(tokenizer.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True))

def evaluate(rows, show=3):
    from collections import defaultdict
    FastLanguageModel.for_inference(model)
    per_cat, wrong = defaultdict(lambda: [0, 0]), []
    for r in rows:
        gold, pred = r["messages"][2]["content"], generate_sql(r["messages"])
        if (g := run(gold)) is None:  # chdb's ClickHouse version rejects the reference query
            print("GOLD FAILS on chdb:", gold[:120])
            continue
        ok = run(pred) == g
        per_cat[r["category"]][0] += ok
        per_cat[r["category"]][1] += 1
        if not ok: wrong.append((r["messages"][1]["content"], pred, gold))
    total, n = sum(c[0] for c in per_cat.values()), sum(c[1] for c in per_cat.values())
    print(f"execution accuracy: {total}/{n} = {total / n:.1%}")
    for cat, (ok, n) in sorted(per_cat.items()):
        print(f"  {cat:10s} {ok}/{n}")
    for q, p, g in wrong[:show]:
        print("\nQ:", q, "\nPRED:\n", p, "\nGOLD:\n", g)

8000



### Baseline (optional, ~5–10 min): the untrained model, for comparison

In [7]:
evaluate(eval_rows)

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

execution accuracy: 10/134 = 7.5%
  agg_group  4/19
  array_map  0/11
  basic      3/9
  cond_agg   0/15
  date       1/14
  join2      1/14
  join3      0/22
  subquery   0/15
  window     1/15

Q: Tampilkan rata-rata nilai deal per stage. 
PRED:
 SELECT 
    deals.stage,
    AVG(deals.value) AS average_value
FROM 
    deals
GROUP BY 
    deals.stage 
GOLD:
 SELECT d.stage, round(avg(d.value), 2) AS rata_rata_nilai FROM deals AS d FINAL GROUP BY d.stage

Q: Jumlah customer per industri ada berapa? 
PRED:
 SELECT industry, COUNT(*) AS customer_count FROM customers GROUP BY industry 
GOLD:
 SELECT c.attributes['industry'] AS industri, count() AS jumlah_customer FROM customers AS c FINAL GROUP BY industri

Q: rekap aktivitas dgn sentimen negatif per tipe 
PRED:
 SELECT 
    activity_type,
    COUNT(*) AS negative_activity_count
FROM 
    activities
WHERE 
    sentiment = 'negative'
GROUP BY 
    activity_type 
GOLD:
 SELECT a.type, count() AS jumlah_aktivitas FROM activities AS a WHERE a

## Train (~2 h on T4 for 3B with ~4,250 samples x 3 epochs, ~1 h for 1.5B)

Each epoch ends with an eval loss on `eval.jsonl`: if it rises from epoch 1 to 2, the model is overfitting and fewer epochs or less data repetition is the fix.

In [8]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

FastLanguageModel.for_training(model)
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=train_ds, eval_dataset=eval_ds,
    args=SFTConfig(
        dataset_text_field="text", max_seq_length=MAX_LEN,
        per_device_train_batch_size=4, gradient_accumulation_steps=2,
        num_train_epochs=EPOCHS, learning_rate=2e-4, warmup_steps=10,
        lr_scheduler_type="linear", optim="adamw_8bit", weight_decay=0.01,
        fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
        eval_strategy="epoch", per_device_eval_batch_size=2,  # eval loss is only reported, it does not change training
        logging_steps=10, seed=3407, output_dir="outputs", report_to="none"))
# Loss only on the SQL, not on the system prompt / question.
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/4256 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/134 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/4256 [00:00<?, ? examples/s]

Map:   0%|          | 0/134 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,256 | Num Epochs = 3 | Total steps = 1,596
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


... (2 outputs left out)

Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-500/tokenizer_config.json.


Filter:   0%|          | 0/134 [00:00<?, ? examples/s]

Unsloth: Not an error, but `num_items_in_batch` could not be counted for Qwen2ForCausalLM's labels, so each micro-batch loss is a mean.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1596/tokenizer_config.json.


TrainOutput(global_step=1596, training_loss=0.06519251660371483, metrics={'train_runtime': 6994.343, 'train_samples_per_second': 1.825, 'train_steps_per_second': 0.228, 'total_flos': 9.320279168242483e+16, 'train_loss': 0.06519251660371483, 'epoch': 3.0})

## Evaluate the fine-tuned model

In [9]:
evaluate(eval_rows, show=5)

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

execution accuracy: 122/134 = 91.0%
  agg_group  17/19
  array_map  11/11
  basic      8/9
  cond_agg   15/15
  date       12/14
  join2      13/14
  join3      21/22
  subquery   13/15
  window     12/15

Q: Cari customer yang punya lebih dari 5 tiket support. 
PRED:
 SELECT c.customer_id, c.name, c.company, c.status, c.segment, count() AS jumlah_tiket FROM support_tickets AS t FINAL INNER JOIN customers AS c FINAL ON t.customer_id = c.customer_id GROUP BY c.customer_id, c.name, c.company, c.status, c.segment HAVING jumlah_tiket > 5 ORDER BY jumlah_tiket DESC 
GOLD:
 SELECT c.customer_id, c.name, count() AS jumlah_tiket FROM support_tickets AS t FINAL INNER JOIN customers AS c FINAL ON t.customer_id = c.customer_id GROUP BY c.customer_id, c.name HAVING jumlah_tiket > 5

Q: Tampilkan kuartil 25, 50, dan 75 nilai deal per stage dalam satu kolom array. 
PRED:
 SELECT d.stage, arraySort([d.value]) AS nilai_deal FROM deals AS d FINAL GROUP BY d.stage ORDER BY d.stage 
GOLD:
 SELECT d.stage

## Export for Ollama

In [10]:
import glob, shutil

model.save_pretrained("lora_adapter"); tokenizer.save_pretrained("lora_adapter")
model.save_pretrained_gguf("gguf", tokenizer, quantization_method="q4_k_m")
gguf = [p for p in glob.glob("gguf*/**/*.gguf", recursive=True) if "Q4_K_M" in p.upper()][0]
shutil.copy(gguf, "text2sql-q4_k_m.gguf")

system = train_rows[0]["messages"][0]["content"]
open("Modelfile", "w", encoding="utf-8").write(f'''FROM ./text2sql-q4_k_m.gguf
TEMPLATE """{{{{- if .System }}}}<|im_start|>system
{{{{ .System }}}}<|im_end|>
{{{{ end }}}}<|im_start|>user
{{{{ .Prompt }}}}<|im_end|>
<|im_start|>assistant
"""
SYSTEM """{system}"""
PARAMETER temperature 0
PARAMETER stop "<|im_end|>"
PARAMETER num_ctx 2048
''')
!ls -lh text2sql-q4_k_m.gguf Modelfile

Unsloth: Restored added_tokens_decoder metadata in lora_adapter/tokenizer_config.json.


Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/764 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


... (9 outputs left out)



Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:35<00:00, 17.86s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:30<00:00, 75.05s/it]


Unsloth: Merge process complete. Saved to `/content/gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b11798-mix-f8a7950 (app-b11798-mix-f8a7950-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['gguf_gguf/Qwen2.5-Coder-3B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Ge

Download (browser download of ~2 GB is slow; the Drive cell below is the alternative). Then locally:
```
ollama create text2sql -f Modelfile
ollama run text2sql "berapa omset bulan ini?"
```

In [12]:
# Alternative: copy to Google Drive
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/text2sql && cp text2sql-q4_k_m.gguf Modelfile /content/drive/MyDrive/text2sql/ && cp -r lora_adapter /content/drive/MyDrive/text2sql/

Mounted at /content/drive
